# ЛР 3.1. Сравнение API Keras: Sequential / Functional / Subclassing

Одинаковая CNN на CIFAR-10, три способа описания модели.


### Цель

Понять различия трёх API Keras при одинаковой архитектуре и сравнить удобство реализации.


In [ ]:
# !pip install tensorflow matplotlib

import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TF', tf.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))


## 1. Данные CIFAR-10 + аугментация


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype(np.int32)
y_test = y_test.squeeze().astype(np.int32)

val_frac = 0.1
n_val = int(len(x_train) * val_frac)
x_val, y_val = x_train[:n_val], y_train[:n_val]
x_train, y_train = x_train[n_val:], y_train[n_val:]

x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

NUM_CLASSES = 10
IMG_SHAPE = (32, 32, 3)
BATCH_SIZE = 128
EPOCHS = 8
LR = 1e-3

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomRotation(0.05),
], name='aug')

print(x_train.shape, x_val.shape, x_test.shape)


## 2. Общая архитектура CNN

Одна и та же сеть будет собрана тремя API.


In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)
    return x


def build_head(x, num_classes=NUM_CLASSES):
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    return layers.Dense(num_classes, activation='softmax')(x)


## 3.1 Sequential API


In [ ]:
def create_sequential():
    model = keras.Sequential([
        layers.Input(shape=IMG_SHAPE),
        data_augmentation,
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax'),
    ])
    return model

model_seq = create_sequential()
model_seq.summary()


## 3.2 Functional API


In [ ]:
def create_functional():
    inputs = keras.Input(shape=IMG_SHAPE)
    x = data_augmentation(inputs)
    x = conv_block(x, 32)
    x = conv_block(x, 64)
    outputs = build_head(x)
    return keras.Model(inputs, outputs, name='functional_cnn')

model_func = create_functional()
model_func.summary()


## 3.3 Subclassing API


In [ ]:
class CnnClassifier(keras.Model):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.aug = data_augmentation
        self.conv1 = layers.Conv2D(32, 3, padding='same', activation='relu')
        self.conv2 = layers.Conv2D(32, 3, padding='same', activation='relu')
        self.pool1 = layers.MaxPooling2D()
        self.drop1 = layers.Dropout(0.25)
        self.conv3 = layers.Conv2D(64, 3, padding='same', activation='relu')
        self.conv4 = layers.Conv2D(64, 3, padding='same', activation='relu')
        self.pool2 = layers.MaxPooling2D()
        self.drop2 = layers.Dropout(0.25)
        self.gap = layers.GlobalAveragePooling2D()
        self.fc = layers.Dense(128, activation='relu')
        self.drop3 = layers.Dropout(0.5)
        self.out = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.aug(inputs, training=training)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)
        x = self.gap(x)
        x = self.fc(x)
        x = self.drop3(x, training=training)
        return self.out(x)

model_sub = CnnClassifier()
_ = model_sub(tf.zeros((1,) + IMG_SHAPE))
model_sub.summary()


## 4. Обучение и сравнение


In [ ]:
def compile_and_train(model, name):
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    t0 = time.time()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )
    elapsed = time.time() - t0
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f'{name}: test_acc={test_acc:.4f}, time={elapsed:.1f}s')
    return history, test_acc, elapsed

results = {}
h_seq, acc_seq, t_seq = compile_and_train(model_seq, 'Sequential')
results['Sequential'] = {'history': h_seq, 'acc': acc_seq, 'time': t_seq}

h_func, acc_func, t_func = compile_and_train(model_func, 'Functional')
results['Functional'] = {'history': h_func, 'acc': acc_func, 'time': t_func}

h_sub, acc_sub, t_sub = compile_and_train(model_sub, 'Subclassing')
results['Subclassing'] = {'history': h_sub, 'acc': acc_sub, 'time': t_sub}


In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history['accuracy'], label='train')
    ax[0].plot(history.history['val_accuracy'], label='val')
    ax[0].set_title(f'{title} accuracy')
    ax[0].legend()
    ax[1].plot(history.history['loss'], label='train')
    ax[1].plot(history.history['val_loss'], label='val')
    ax[1].set_title(f'{title} loss')
    ax[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(h_seq, 'Sequential')
plot_history(h_func, 'Functional')
plot_history(h_sub, 'Subclassing')

print('{:<14}{:>10}{:>10}'.format('API', 'Accuracy', 'Time, s'))
for name in ['Sequential', 'Functional', 'Subclassing']:
    r = results[name]
    print('{:<14}{:>10.4f}{:>10.1f}'.format(name, r['acc'], r['time']))


**Вопрос 1.** Почему аугментация помогает бороться с переобучением?

**Ответ:**

---

**Вопрос 2.** Когда Functional API предпочтительнее Sequential?

**Ответ:**

---

**Вопрос 3.** Когда оправдан Subclassing API?

**Ответ:**
